# 3. Deploy your segmentation model

In [1]:
#Import libraries
import os
from model_class import ModelSegmentation
import pickle
from pictures_class import Pictures
import pandas as pd
#Inputs
working_directory=r"C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper"
pictures_directory=os.path.join(working_directory, "pics_cortas_rehechas_shell")
model_path=os.path.join(working_directory, "models/hyper_rgb_entrenado_shell.pt")
info_data_completed_path=os.path.join(working_directory, "info_data_completed_shell_2024.txt")
info_data_completed=pd.read_csv(info_data_completed_path,sep="\t")
sam_path=os.path.join(working_directory, "models/sam2.1_l.pt")

## Choose your reconstruction approach and measure (almond)

### Slice predict reconstruct

In [ ]:
# Join patches approach

model=ModelSegmentation(working_directory=working_directory)
masks=model.slice_predict_reconstruct(input_folder=pictures_directory,imgsz=1280, model_path=model_path,
                                          slice_height=1280, slice_width=1280,overlap_height_ratio=0.2,
                                          overlap_width_ratio=0.2, conf=0.00001)

In [ ]:
## Example with slice predict reconstruct approach
pictures_object=Pictures(working_directory=working_directory, input_folder=pictures_directory,info_file=info_data_completed,
                      fruit="apple", binary_masks=True, project_name="coin_shell_2023", blurring_binary_masks=False)
pictures_object.set_postsegmentation_parameters(sahi=False, segmentation_input=masks, smoothing=False, smoothing_iterations=2, kernel_smoothing=3,
                        watershed=True, kernel_watershed=5, threshold_watershed=0.6)
pictures_object.measure_almonds(margin=400)

# Save
with open(f'{working_directory}/pictures_object_watershed.pkl', 'wb') as file:
    pickle.dump(pictures_object, file)

### SAHI

In [ ]:
model=ModelSegmentation(working_directory=working_directory)
masks=model.predict_model_sahi(model_path=model_path, check_result=False, folder_input=pictures_directory,
                                            retina_masks=True,
                                              postprocess_match_threshold=0.1, overlap_height_ratio=0.4,
                                                overlap_width_ratio=0.4, postprocess_match_metric="IOU", 
                                                postprocess_type="GREEDYNMM", slice_height=1280, slice_width=1280,
                                                  confidence_treshold=0.01,
                                                  imgsz=1280)

In [ ]:
## Example with SAHI approach
pictures_object=Pictures(working_directory=working_directory, input_folder=pictures_directory,info_file=info_data_completed,
                      fruit="Seed", binary_masks=True, project_name="seed_develop_v1",  blurring_binary_masks=False)
pictures_object.set_postsegmentation_parameters(sahi=True, segmentation_input=masks)
pictures_object.measure_almonds(margin=400)

# Guardar el objeto en un archivo
with open(f'{working_directory}/pictures_object_sahi.pkl', 'wb') as file:
    pickle.dump(pictures_object, file)

### Two STEP YOLO + SAM

In [3]:
model=ModelSegmentation(working_directory=working_directory)

masks=model.predict_detection_sam(model_path=model_path,folder_input=pictures_directory,
                                   sam_path=sam_path,
        imgsz=1280,
        conf=0.25,
        max_det=3000,
        save_results=True,
        output_name="sam_results_todo", retina_masks=True, margin_bbox=0.05, max_dimension=5000, nms_largeimages=0.5)


Detected GPU: NVIDIA GeForce RTX 3060
Total GPU Memory: 12.00 GB

0: 1280x1280 3 shells, 60.2ms
1: 1280x1280 3 shells, 60.2ms
Speed: 6.8ms preprocess, 60.2ms inference, 8.1ms postprocess per image at shape (1, 3, 1280, 1280)
C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\pics_cortas_rehechas_shell\rgb_50_HYP_shell_shell_281125_1_2025-11-28-10-26-12_0.jpg

0: 1024x1024 1 0, 1 1, 1 2, 750.9ms
Speed: 6.0ms preprocess, 750.9ms inference, 20.5ms postprocess per image at shape (1, 3, 1024, 1024)
C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\pics_cortas_rehechas_shell\rgb_50_HYP_shell_shell_281125_2_2025-11-28-11-26-20_4.jpg

0: 1024x1024 1 0, 1 1, 1 2, 404.4ms
Speed: 4.2ms preprocess, 404.4ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 1024)
âœ… Finished. Results saved in: C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\sam_results_todo


In [4]:
masks_revised=model.check_segmentation_twosteps(sam_path=sam_path)

Starting check_segmentation_twosteps...
Loading SAM model...
SAM model loaded.
Loading refiner...
Refiner loaded.
Loading discard_morphology from: C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\discard_morphology_session.txt
Processing image 1/2: C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\pics_cortas_rehechas_shell\rgb_50_HYP_shell_shell_281125_1_2025-11-28-10-26-12_0.jpg
Selected action: None, Points: []
Saving changes and moving to the next image...
Clean TXT updated: C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\discard_morphology_session.txt
Processing image 1/2: C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\pics_cortas_rehechas_shell\rgb_50_HYP_shell_shell_281125_1_2025-11-28-10-26-12_0.jpg
Selected action: None, Points: []
Saving changes and moving to the next image...
Clean TXT updated: C:\Users\Pheno\Documents\database_almondcv2\modelos_rgb_hyper\discard_morphology_session.txt


In [ ]:
model=ModelSegmentation(working_directory=working_directory)
with open(f"{working_directory}/all_results_backup.pkl", "rb") as f:
    model.all_results = pickle.load(f)
masks_revised = model.check_segmentation_twosteps(sam_path=sam_path)

In [5]:
## Example with two_step approach
pictures_object=Pictures(working_directory=working_directory, input_folder=pictures_directory,info_file=info_data_completed,
                      fruit="Shell", binary_masks=True, project_name="Fenotipado",  blurring_binary_masks=False)
pictures_object.set_postsegmentation_parameters(two_step=True, segmentation_input=masks_revised)
pictures_object.measure_almonds(margin=0.5)

# Guardar el objeto en un archivo
with open(f'{working_directory}/objetct_after_measurement.pkl', 'wb') as file:
    pickle.dump(pictures_object, file)

rgb_50_HYP_shell_shell_281125_1_2025-11-28-10-26-12_0.jpg
rgb_50_HYP_shell_shell_281125_2_2025-11-28-11-26-20_4.jpg


c:\Users\Pheno\Desktop\Almond_CV\almondcv2\pictures_class.py:573: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  morphology_table = pd.concat([morphology_table, row], ignore_index=True)
c:\Users\Pheno\Desktop\Almond_CV\almondcv2\pictures_class.py:587: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  general_table=pd.concat([general_table,row_general], ignore_index=True)
c:\Users\Pheno\Desktop\Almond_CV\almondcv2\pictures_class.py:647: SettingWithCopyWarning: 
A value is trying to be set on a copy of

## Choose your reconstruction approach and measure (general)

### Slice predict reconstruct

In [ ]:
pictures_object.info_file["Pixelmetric"]

In [ ]:
# Join patches approach

model=ModelSegmentation(working_directory=working_directory)
masks=model.slice_predict_reconstruct(input_folder=pictures_directory,imgsz=640, model_path=model_path,
                                          slice_height=640, slice_width=640,overlap_height_ratio=0.2,
                                          overlap_width_ratio=0.2, conf=0.01)

In [ ]:
## Example with slice predict reconstruct approach
pictures_object=Pictures(working_directory=working_directory, input_folder=pictures_directory,info_file=info_data_completed,
                      fruit="apple", binary_masks=True, project_name="apple", blurring_binary_masks=False)
pictures_object.set_postsegmentation_parameters(sahi=False, segmentation_input=masks, smoothing=False, smoothing_iterations=2, kernel_smoothing=3,
                        watershed=True, kernel_watershed=5, threshold_watershed=0.6)
pictures_object.measure_general(margin=400)

# Save
with open(f'{working_directory}/pictures_object_watershed.pkl', 'wb') as file:
    pickle.dump(pictures_object, file)

### SAHI

In [ ]:
model=ModelSegmentation(working_directory=working_directory)
masks=model.predict_model_sahi(model_path=model_path, check_result=False, folder_input=pictures_directory,
                                            retina_masks=True,
                                              postprocess_match_threshold=0.05, overlap_height_ratio=0.2,
                                                overlap_width_ratio=0.2, postprocess_match_metric="IOS", 
                                                postprocess_type="GREEDYNMM", slice_height=640, slice_width=640,
                                                  confidence_treshold=0.6,
                                                  imgsz=640)

In [ ]:
## Example with SAHI approach
pictures_object=Pictures(working_directory=working_directory, input_folder=pictures_directory,info_file=info_data_completed,
                      fruit="apple_sahi", binary_masks=True, project_name="apple_sahi",  blurring_binary_masks=False)
pictures_object.set_postsegmentation_parameters(sahi=True, segmentation_input=masks)
pictures_object.measure_general(margin=400)

# Guardar el objeto en un archivo
with open(f'{working_directory}/pictures_object_sahi.pkl', 'wb') as file:
    pickle.dump(pictures_object, file)